# Prep data set to pass off for training

### Creating 3 datasets
1) Text = Metadata only, 2) Text = Transcript, 3) Text = Title, Description, Transcript 

In [3]:
import pandas as pd

In [9]:
input = r"../../data/mccray/october_sprint/mccray d_main.csv"
df = pd.read_csv(input)
list(df)

['Title',
 'Creator',
 'Contributors',
 'Date',
 'Approximate Date',
 'Source',
 'Subject',
 'Local Subject',
 'S.C. County',
 'Description',
 'Extent',
 'Digital Collection',
 'Website',
 'Contributing Institution',
 'Rights',
 'Time Period',
 'Geographic Location',
 'Language',
 'Digitization Specifications',
 'Date Digital',
 'Type',
 'Format',
 'Media Type',
 'Identifier',
 'Note',
 'Digital Assistant',
 'OCLC number',
 'Date created',
 'Date modified',
 'Reference URL',
 'CONTENTdm number',
 'CONTENTdm file name',
 'CONTENTdm file path',
 'Year',
 'Original Transcript',
 'Original Len',
 'Special Pattern',
 'General Pattern',
 'Repeat Chars',
 'Short/No Transcript',
 'Quality',
 'Issue Types',
 'Detected Artifacts',
 'Semi-clean Transcript',
 '% English']

In [10]:
# Remove rows if Semi-clean Transcript length < 30
df_filtered = df[df['Semi-clean Transcript'].str.len() >= 30].copy()
print(f"Filtered from {len(df)} to {len(df_filtered)} rows (removed {len(df) - len(df_filtered)} short transcripts)")

# Helper function to create formatted text field
def create_text_field(row, fields):
    """Create formatted text field from specified columns"""
    text_parts = []
    for field in fields:
        if field in row and pd.notna(row[field]) and str(row[field]).strip():
            # Format as [Field]: "content"
            content = str(row[field]).strip()
            text_parts.append(f"[{field}]: \"{content}\"")
    return "\n".join(text_parts)

# 1. Title, Description, and Date Fields only
title_description = df_filtered.copy()
title_description['text'] = title_description.apply(
    lambda row: create_text_field(row, ['Title', 'Description', 'Date']), axis=1
)
# Keep only essential columns
title_description = title_description[['CONTENTdm number', 'Year', 'text']].copy()

# 2. Semi-clean Transcript field (exclude [Field name] formatting)
semiclean = df_filtered.copy()
semiclean['text'] = semiclean['Semi-clean Transcript'].fillna('').astype(str)
# Keep only essential columns
semiclean = semiclean[['CONTENTdm number', 'Year', 'text']].copy()

# 3. Title, Description, Date, and Semi-clean Transcript Fields (with Date before Transcript)
all_meta = df_filtered.copy()
# Create the combined text field with Date before Transcript
all_meta['text'] = all_meta.apply(
    lambda row: create_text_field(row, ['Title', 'Description', 'Date']) + 
                f"\n[Transcript]: \"{str(row['Semi-clean Transcript']).strip()}\"" 
                if pd.notna(row['Semi-clean Transcript']) and str(row['Semi-clean Transcript']).strip() 
                else create_text_field(row, ['Title', 'Description', 'Date']), 
    axis=1
)
# Keep only essential columns
all_meta = all_meta[['CONTENTdm number', 'Year', 'text']].copy()

# Display sample outputs
print("\n=== SAMPLE OUTPUTS ===")
print(f"\n1. Title + Description + Date dataset: {len(title_description)} rows")
print("Sample:")
print(title_description['text'].iloc[0][:200] + "...")

print(f"\n2. Transcript only dataset: {len(semiclean)} rows")
print("Sample:")
print(semiclean['text'].iloc[0][:200] + "...")

print(f"\n3. All metadata dataset: {len(all_meta)} rows")
print("Sample:")
print(all_meta['text'].iloc[0][:300] + "...")

# Save each dataset
output_dir = "../../data/mccray/training_datasets/"
import os
os.makedirs(output_dir, exist_ok=True)

title_description.to_csv(f"{output_dir}mccray_title_description_date.csv", index=False)
semiclean.to_csv(f"{output_dir}mccray_transcript_only.csv", index=False)
all_meta.to_csv(f"{output_dir}mccray_all_metadata.csv", index=False)

print(f"\n=== DATASETS SAVED ===")
print(f"1. Title + Description + Date: {output_dir}mccray_title_description_date.csv")
print(f"2. Transcript only: {output_dir}mccray_transcript_only.csv")
print(f"3. All metadata: {output_dir}mccray_all_metadata.csv")

# Additional step: Filter for 1940-1950s
print(f"\n=== FILTERING FOR 1940s ===")
def filter_1940s(df):
    # Convert Year to numeric, handle any non-numeric values
    df_copy = df.copy()
    df_copy['Year'] = pd.to_numeric(df_copy['Year'], errors='coerce')
    # Filter for 1940-1950
    return df_copy[(df_copy['Year'] >= 1940) & (df_copy['Year'] <= 1950)]

title_description_1940s = filter_1940s(title_description)
semiclean_1940s = filter_1940s(semiclean)
all_meta_1940s = filter_1940s(all_meta)

print(f"1940s filtering results:")
print(f"  Title + Description + Date: {len(title_description)} → {len(title_description_1940s)} rows")
print(f"  Transcript only: {len(semiclean)} → {len(semiclean_1940s)} rows")
print(f"  All metadata: {len(all_meta)} → {len(all_meta_1940s)} rows")

# Save 1940s datasets
title_description_1940s.to_csv(f"{output_dir}mccray_title_description_date_1940s.csv", index=False)
semiclean_1940s.to_csv(f"{output_dir}mccray_transcript_only_1940s.csv", index=False)
all_meta_1940s.to_csv(f"{output_dir}mccray_all_metadata_1940s.csv", index=False)

print(f"\n=== 1940s DATASETS SAVED ===")
print(f"1. Title + Description + Date (1940s): {output_dir}mccray_title_description_date_1940s.csv")
print(f"2. Transcript only (1940s): {output_dir}mccray_transcript_only_1940s.csv")
print(f"3. All metadata (1940s): {output_dir}mccray_all_metadata_1940s.csv")

# Display final statistics
print(f"\n=== FINAL STATISTICS ===")
print(f"Original dataset: {len(df)} rows")
print(f"After filtering (transcript >= 30 chars): {len(df_filtered)} rows")
print(f"Average text length by dataset:")
print(f"  Title + Description + Date: {title_description['text'].str.len().mean():.0f} chars")
print(f"  Transcript only: {semiclean['text'].str.len().mean():.0f} chars")
print(f"  All metadata: {all_meta['text'].str.len().mean():.0f} chars")
print(f"1940s subset text lengths:")
print(f"  Title + Description + Date (1940s): {title_description_1940s['text'].str.len().mean():.0f} chars")
print(f"  Transcript only (1940s): {semiclean_1940s['text'].str.len().mean():.0f} chars")
print(f"  All metadata (1940s): {all_meta_1940s['text'].str.len().mean():.0f} chars")

Filtered from 11820 to 11694 rows (removed 126 short transcripts)

=== SAMPLE OUTPUTS ===

1. Title + Description + Date dataset: 11694 rows
Sample:
[Title]: "Afro-American Newsboy Application signed by Mrs. C. B. Berry"
[Description]: "An application to be a Newsboy for the Afro-American submitted by Mrs. C. B. Berry."...

2. Transcript only dataset: 11694 rows
Sample:
-5491 AFRO-AMERICAN NEWSBOY'S APPLICATION I hereby apply for membership in the AFRO Newsboys' Association of Carriers and Street Salesmen, and if accepted, this application being properly signed by my...

3. All metadata dataset: 11694 rows
Sample:
[Title]: "Afro-American Newsboy Application signed by Mrs. C. B. Berry"
[Description]: "An application to be a Newsboy for the Afro-American submitted by Mrs. C. B. Berry."
[Transcript]: "-5491 AFRO-AMERICAN NEWSBOY'S APPLICATION I hereby apply for membership in the AFRO Newsboys' Association of Car...

=== DATASETS SAVED ===
1. Title + Description + Date: ../../data/mccray/t